# SpendShield Deterministic Candidate Feature Matrix

This notebook creates a research-only candidate feature matrix from the validated synthetic transaction files.

## Objective

Audit prior-only history features, separate targets from inputs, fit preprocessing on training rows only, and write deterministic temporal feature artifacts.

## Non-goals

No database access, payment behavior, API, fraud decision, production model, or real-world claim is made.

## Provenance and target boundary

The source is the isolated `synthetic_research` dataset. `scenario_label` is retained as `target_scenario_label` for audit only and excluded from model feature columns. These are generated scenario labels, not real fraud labels.

In [1]:
from pathlib import Path
import sys
REPOSITORY_ROOT = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "ml" / "synthetic_dataset_generator.py").exists())
sys.path.insert(0, str(REPOSITORY_ROOT))
SOURCE_DIR = REPOSITORY_ROOT / "data" / "synthetic"
FEATURE_DIR = SOURCE_DIR / "features"
import json
from ml.feature_matrix import AUDIT_COLUMNS, CATEGORICAL_SOURCE_FIELDS, CANDIDATE_FEATURE_FIELDS, EXCLUDED_FIELDS, NUMERIC_SOURCE_FIELDS, TARGET_COLUMN, audit_historical_features, build_feature_artifact, validate_feature_artifact
source_manifest = json.loads((SOURCE_DIR / 'dataset_manifest.json').read_text(encoding='utf-8'))
{'dataset_type': source_manifest['dataset_type'], 'dataset_version': source_manifest['dataset_version'], 'row_count': source_manifest['row_count'], 'split_counts': source_manifest['split_counts'], 'temporal_rule': source_manifest['split_rule']['method']}

{'dataset_type': 'synthetic_research',
 'dataset_version': 'v1',
 'row_count': 10000,
 'split_counts': {'test': 1458, 'train': 7061, 'validation': 1481},
 'temporal_rule': 'timestamp_threshold'}

## Feature eligibility table

Only fields available at transaction time or computed from earlier records are eligible. Identifiers remain audit columns and are not model inputs.

In [2]:
eligibility = [{'feature': 'amount', 'type': 'Numeric', 'allowed': True, 'reason': 'Available at transaction time'}, {'feature': 'transaction_hour', 'type': 'Numeric', 'allowed': True, 'reason': 'Available at transaction time'}, {'feature': 'merchant_category', 'type': 'Categorical', 'allowed': True, 'reason': 'Available at transaction time'}, {'feature': 'user_historical_average_amount_before', 'type': 'Numeric', 'allowed': True, 'reason': 'Earlier records only'}, {'feature': 'scenario_label', 'type': 'Target', 'allowed': False, 'reason': 'Label leakage'}, {'feature': 'scenario_injection_reason', 'type': 'Metadata', 'allowed': False, 'reason': 'Direct target leakage'}, {'feature': 'future_transaction_count', 'type': 'Numeric', 'allowed': False, 'reason': 'Future leakage'}]
eligibility

[{'feature': 'amount',
  'type': 'Numeric',
  'allowed': True,
  'reason': 'Available at transaction time'},
 {'feature': 'transaction_hour',
  'type': 'Numeric',
  'allowed': True,
  'reason': 'Available at transaction time'},
 {'feature': 'merchant_category',
  'type': 'Categorical',
  'allowed': True,
  'reason': 'Available at transaction time'},
 {'feature': 'user_historical_average_amount_before',
  'type': 'Numeric',
  'allowed': True,
  'reason': 'Earlier records only'},
 {'feature': 'scenario_label',
  'type': 'Target',
  'allowed': False,
  'reason': 'Label leakage'},
 {'feature': 'scenario_injection_reason',
  'type': 'Metadata',
  'allowed': False,
  'reason': 'Direct target leakage'},
 {'feature': 'future_transaction_count',
  'type': 'Numeric',
  'allowed': False,
  'reason': 'Future leakage'}]

In [3]:
import csv
with (SOURCE_DIR / 'spendshield_synthetic_transactions_v1.csv').open(newline='', encoding='utf-8') as handle:
    source_rows = tuple(csv.DictReader(handle))
history_audit = audit_historical_features(source_rows)
assert history_audit['valid'], history_audit
history_audit

{'valid': True,
 'errors': [],
 'rows_audited': 10000,
 'tie_breaking': 'timestamp ascending, then synthetic_transaction_id ascending',
 'first_transaction_policy': {'historical_count': 0,
  'historical_average': '0.00 fallback',
  'time_since_previous': 'missing and imputed from training median for model input'}}

## Build and validate the artifact

Median imputation and one-hot vocabularies are fitted using training rows only. Existing timestamp-then-transaction-ID ordering is used for prior-only history. Unknown categorical values have an explicit bucket.

In [4]:
feature_manifest = build_feature_artifact(SOURCE_DIR, FEATURE_DIR)
feature_validation = validate_feature_artifact(FEATURE_DIR)
assert feature_validation['valid'], feature_validation
{'feature_count': feature_validation['feature_count'], 'feature_names': feature_validation['feature_names'], 'split_counts': feature_validation['split_counts'], 'excluded_features_present': feature_validation['excluded_fields_present_as_features']}

{'feature_count': 28,
 'feature_names': ['amount',
  'transaction_hour',
  'day_of_week',
  'user_historical_transaction_count_before',
  'user_historical_average_amount_before',
  'time_since_previous_transaction_seconds',
  'user_historical_category_frequency_before',
  'time_since_previous_transaction_seconds__missing',
  'currency__INR',
  'currency____unknown',
  'merchant_category____unknown',
  'merchant_category__bills',
  'merchant_category__education',
  'merchant_category__entertainment',
  'merchant_category__food',
  'merchant_category__grocery',
  'merchant_category__healthcare',
  'merchant_category__home',
  'merchant_category__other',
  'merchant_category__shopping',
  'merchant_category__subscriptions',
  'merchant_category__transport',
  'merchant_category__travel',
  'transaction_channel__BANK_SIMULATED',
  'transaction_channel__CARD_SIMULATED',
  'transaction_channel__QR_SIMULATED',
  'transaction_channel__WALLET_SIMULATED',
  'transaction_channel____unknown'],
 's

In [5]:
with (FEATURE_DIR / 'candidate_features_train.csv').open(newline='', encoding='utf-8') as handle:
    train_rows = tuple(csv.DictReader(handle))
numeric_stats = {name: {'minimum': min(float(row[name]) for row in train_rows), 'maximum': max(float(row[name]) for row in train_rows)} for name in NUMERIC_SOURCE_FIELDS}
{'audit_columns': list(AUDIT_COLUMNS), 'target_column': TARGET_COLUMN, 'numeric_stats_from_train': numeric_stats, 'categorical_source_fields': list(CATEGORICAL_SOURCE_FIELDS), 'candidate_source_fields': list(CANDIDATE_FEATURE_FIELDS), 'excluded_source_fields': list(EXCLUDED_FIELDS)}

{'audit_columns': ['synthetic_transaction_id',
  'synthetic_user_id',
  'timestamp',
  'dataset_split'],
 'target_column': 'target_scenario_label',
 'numeric_stats_from_train': {'amount': {'minimum': 71.94,
   'maximum': 14574.67},
  'transaction_hour': {'minimum': 0.0, 'maximum': 23.0},
  'day_of_week': {'minimum': 0.0, 'maximum': 6.0},
  'user_historical_transaction_count_before': {'minimum': 0.0,
   'maximum': 18.0},
  'user_historical_average_amount_before': {'minimum': 0.0,
   'maximum': 8956.49},
  'time_since_previous_transaction_seconds': {'minimum': 62.0,
   'maximum': 2517707.0},
  'user_historical_category_frequency_before': {'minimum': 0.0,
   'maximum': 1.0}},
 'categorical_source_fields': ['currency',
  'merchant_category',
  'transaction_channel'],
 'candidate_source_fields': ['amount',
  'currency',
  'transaction_hour',
  'day_of_week',
  'merchant_category',
  'transaction_channel',
  'user_historical_transaction_count_before',
  'user_historical_average_amount_before

## Conclusion and limitations

The matrix is deterministic, temporally partitioned, and leakage-audited. It is not production-ready. The next notebook evaluates a fixed research baseline against the synthetic scenario target only.